# 04 · Recommendations & Business Analytics

End-to-end recommendations, the ablation study, campaign analytics and the A/B toolkit.

> Synthetic data — methodology demo, not CARS24 results. Run `make pipeline-small` first so artifacts exist.

In [ ]:
from driveintent.config import load_config
from driveintent.models.ranking import recommend_for_user
import pandas as pd
cfg = load_config()
ev = pd.read_parquet(cfg.raw_data/'events.parquet')
uid = ev['user_id'].value_counts().index[0]
out = recommend_for_user(cfg, uid, limit=6)
print('intent:', out['intent_summary'])
for r in out['recommendations'][:3]:
    print(r['rank'], r['make'], r['model'], '| deal', r['deal_score'], '|', r['reasons'][:2])

## Ranking strategy comparison & ablation

In [ ]:
from driveintent.analytics.analytics import ablation_study
import json
rm = json.loads((cfg.artifacts/'metrics'/'ranking_metrics.json').read_text())
print({k: round(v['ndcg_at_10'],3) for k,v in rm.items() if isinstance(v,dict)})
ablation_study(cfg)

## Campaign lead quality & budget simulation

In [ ]:
from driveintent.analytics.analytics import campaign_performance, budget_optimizer
campaign_performance(cfg)[['campaign_name','channel','leads','bookings',
    'expected_lead_value','expected_roas']]

In [ ]:
budget_optimizer(cfg)

## A/B test design

In [ ]:
from driveintent.analytics.analytics import sample_size_two_proportions, simulate_experiment
print('per-arm n (3.5% base, 15% MDE):', sample_size_two_proportions(0.035, 0.15))
simulate_experiment(cfg, n_per_arm=5000, true_lift=0.15)